In [8]:
from pathlib import Path
from functools import reduce

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

In [10]:
# Load and merge files
ELSA_DIR = Path("../data/elsa")
files = sorted(ELSA_DIR.glob("*.csv"))

dfs = []
for f in files:
    df = pd.read_csv(f)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df["idauniq"] = pd.to_numeric(df["idauniq"], errors="coerce").astype("Int64")
    dfs.append(df.drop_duplicates(subset="idauniq"))

base, *rest = dfs
rest = [d.drop(columns=["hhid", "xwgt", "gor"], errors="ignore") for d in rest]

df = reduce(lambda l, r: pd.merge(l, r, on="idauniq", how="outer"), [base] + rest)
df["non_homeowner"] = 1 - df["homeowner"]
print(df.shape)

(5000, 35)


In [11]:
age_cols = ["age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85_plus"]
sex_cols = ["male", "female"]
partner_cols = ["partner_has_partner", "partner_does_not_have_partner"]
homeowner_cols = ["homeowner", "non_homeowner"]
wealth_cols = ["wealth_q1", "wealth_q2", "wealth_q3", "wealth_q4", "wealth_q5"]
health_cols = ["health_very_good_health", "health_good_health",
               "health_fair_health", "health_bad_health"]
nssec_cols = ["nssec_managerial", "nssec_intermediate", "nssec_manual"]
eth_cols = ["ethnicity_white", "ethnicity_non_white"]
born_cols = ["uk_born", "non_uk_born"]

In [12]:
# Define labels 

def to_labels(frame, cols):
    out = pd.Series(pd.NA, index=frame.index, dtype="object")
    for col in cols:
        out[frame[col] == 1] = col
    return out


def collapse(frame):
    return pd.DataFrame({
        "age": to_labels(frame, age_cols),
        "sex": to_labels(frame, sex_cols),
        "partner": to_labels(frame, partner_cols),
        "homeowner": to_labels(frame, homeowner_cols),
        "wealth": to_labels(frame, wealth_cols),
        "health": to_labels(frame, health_cols),
        "ethnicity": to_labels(frame, eth_cols),
        "country_of_birth": to_labels(frame, born_cols),
        "nssec": to_labels(frame, nssec_cols),
    })


lab = collapse(df)
assert lab[["age", "sex", "partner"]].notna().all().all()

missing = lab.isna()
print(pd.DataFrame({
    "n_missing": missing.sum(),
    "pct_missing": (100 * missing.mean()).round(1),
}))

                  n_missing  pct_missing
age                       0          0.0
sex                       0          0.0
partner                   0          0.0
homeowner                42          0.8
wealth                  117          2.3
health                  170          3.4
ethnicity                33          0.7
country_of_birth         17          0.3
nssec                  1558         31.2


In [13]:
# Build imputation model

def impute_logit(lab, target, predictors, rng):
    miss = lab[target].isna()
    if not miss.any():
        return lab[target].copy(), pd.DataFrame()

    X_obs = lab.loc[~miss, predictors]
    y_obs = lab.loc[~miss, target]
    boot = rng.choice(len(X_obs), size=len(X_obs), replace=True)

    model = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        LogisticRegression(max_iter=1000),
    )
    model.fit(X_obs.iloc[boot], y_obs.iloc[boot])

    probs = model.predict_proba(lab.loc[miss, predictors])
    classes = model.classes_
    u = rng.random(len(probs))
    pick = np.minimum((probs.cumsum(axis=1) < u[:, None]).sum(axis=1), len(classes) - 1)

    filled = lab[target].copy()
    filled[miss] = classes[pick]

    probs = pd.DataFrame(probs, columns=[f"p_{c}" for c in classes], index=lab.index[miss])
    return filled, probs


def make_completed(df, rng):
    lab = collapse(df)

    logs = []
    for var, preds in order.items():
        lab[var], probs = impute_logit(lab, var, preds, rng)
        probs["imputed"] = lab.loc[probs.index, var]
        probs.insert(0, "variable", var)
        probs.insert(0, "idauniq", df.loc[probs.index, "idauniq"])
        logs.append(probs)

    d = df.copy()
    for var, cols in [("health", health_cols), ("nssec", nssec_cols),
                      ("ethnicity", eth_cols), ("country_of_birth", born_cols)]:
        for col in cols:
            d[col] = (lab[var] == col).astype(int)

    sms = d[["idauniq"] + constraints + outcomes].copy()
    sms[constraints] = sms[constraints].astype(int)

    for name, cols in blocks.items():
        assert (sms[cols].sum(axis=1) == 1).all(), name

    return sms, lab, pd.concat(logs)

In [19]:
# Impute sequentially 
order = {}
preds = ["age", "sex", "partner"] # complete variables
for var in ["country_of_birth", "ethnicity", "homeowner", "wealth", "health", "nssec"]:
    order[var] = preds.copy()
    preds.append(var)

constraints = age_cols + sex_cols + partner_cols + nssec_cols + health_cols + eth_cols + born_cols
outcomes = ["has_any_care_needs"]

blocks = {"age": age_cols, "sex": sex_cols, "partner": partner_cols, "nssec": nssec_cols,
          "health": health_cols, "ethnicity": eth_cols, "country_of_birth": born_cols}

out_dir = Path("../data/imputed_elsa")
M = 20
rng = np.random.default_rng()

completed, labs, logs = {}, {}, []
for m in range(1, M + 1):
    sms, lab_m, log = make_completed(df, rng)
    sms.to_csv(out_dir / f"sms_imputed_m{m:02d}.csv", index=False)

    completed[m] = sms
    labs[m] = lab_m
    logs.append(log.assign(m=m))

pd.concat(logs).to_csv(out_dir / "imputation_log.csv", index=False)

In [22]:
# Masked validation
missing_before = collapse(df).isna()

results = {}
for var, preds in order.items():
    known = labs[1][~missing_before[var]]
    frac = 0.31 if var == "nssec" else 0.10

    n, acc, err = [], [], []
    for _ in range(M):
        hide = rng.random(len(known)) < frac
        test = known.copy()
        test.loc[hide, var] = pd.NA
        guess, _ = impute_logit(test, var, preds, rng)

        truth, guess = known.loc[hide, var], guess[hide]
        gap = truth.value_counts(normalize=True).sub(guess.value_counts(normalize=True), fill_value=0)

        n.append(hide.sum())
        acc.append((truth == guess).mean())
        err.append(gap.abs().mean())

    p = known[var].value_counts(normalize=True)
    results[var] = {
        "n": np.mean(n),
        "accuracy": np.mean(acc),
        "sd": np.std(acc, ddof=1),
        "random": (p ** 2).sum(),
        "dist_error": np.mean(err),
    }

pd.DataFrame(results).T.round(3)

,n,accuracy,sd,random,dist_error
country_of_birth,503.05,0.740,0.017,0.741,0.018
ethnicity,495.35,0.917,0.012,0.893,0.009
homeowner,487.85,0.792,0.016,0.772,0.018
wealth,492.45,0.252,0.022,0.210,0.023
health,477.60,0.323,0.022,0.286,0.023
nssec,1067.20,0.404,0.011,0.354,0.018
